In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load dataset from local path

csv_path = r"C:\Users\12162\Downloads\jena_climate_2009_2016.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (420551, 15)
             Date Time  p (mbar)  T (degC)  Tpot (K)  Tdew (degC)  rh (%)  \
0  01.01.2009 00:10:00    996.52     -8.02    265.40        -8.90    93.3   
1  01.01.2009 00:20:00    996.57     -8.41    265.01        -9.28    93.4   
2  01.01.2009 00:30:00    996.53     -8.51    264.91        -9.31    93.9   
3  01.01.2009 00:40:00    996.51     -8.31    265.12        -9.07    94.2   
4  01.01.2009 00:50:00    996.51     -8.27    265.15        -9.04    94.1   

   VPmax (mbar)  VPact (mbar)  VPdef (mbar)  sh (g/kg)  H2OC (mmol/mol)  \
0          3.33          3.11          0.22       1.94             3.12   
1          3.23          3.02          0.21       1.89             3.03   
2          3.21          3.01          0.20       1.88             3.02   
3          3.26          3.07          0.19       1.92             3.08   
4          3.27          3.08          0.19       1.92             3.09   

   rho (g/m**3)  wv (m/s)  max. wv (m/s)  wd (deg)  
0    

In [ ]:

# 2. Prepare the data

# Drop the date column
raw_data = df.iloc[:, 1:].values.astype("float32")

# Temperature target column: T (degC)
temperature = raw_data[:, 1]

num_samples = len(raw_data)
num_train_samples = int(0.5 * num_samples)
num_val_samples = int(0.25 * num_samples)
num_test_samples = num_samples - num_train_samples - num_val_samples

print("Train samples:", num_train_samples)
print("Validation samples:", num_val_samples)
print("Test samples:", num_test_samples)

# Normalize using only training data
mean = raw_data[:num_train_samples].mean(axis=0)
std = raw_data[:num_train_samples].std(axis=0)
raw_data = (raw_data - mean) / std

Train samples: 210275
Validation samples: 105137
Test samples: 105139


In [ ]:

# 3. Build time-series datasets

sampling_rate = 6       # one point per hour
sequence_length = 120   # 5 days
delay = sampling_rate * (sequence_length + 24 - 1)
batch_size = 256

def make_dataset(start_index, end_index, shuffle=False):
    dataset = keras.utils.timeseries_dataset_from_array(
        data=raw_data[:-delay],
        targets=temperature[delay:],
        sampling_rate=sampling_rate,
        sequence_length=sequence_length,
        shuffle=shuffle,
        batch_size=batch_size,
        start_index=start_index,
        end_index=end_index
    )
    return dataset

train_dataset = make_dataset(
    start_index=0,
    end_index=num_train_samples,
    shuffle=True
)

val_dataset = make_dataset(
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples,
    shuffle=False
)

test_dataset = make_dataset(
    start_index=num_train_samples + num_val_samples,
    end_index=None,
    shuffle=False
)


In [ ]:

# 4. Build models

def build_gru_model():
    inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
    x = layers.GRU(32)(inputs)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

def build_stacked_gru_model():
    inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
    x = layers.GRU(32, return_sequences=True)(inputs)
    x = layers.GRU(32)(x)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

def build_lstm_model():
    inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
    x = layers.LSTM(32, return_sequences=True)(inputs)
    x = layers.LSTM(32)(x)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

def build_conv_gru_model():
    inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
    x = layers.Conv1D(32, 5, activation="relu")(inputs)
    x = layers.MaxPooling1D(3)(x)
    x = layers.GRU(32)(x)
    outputs = layers.Dense(1)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model


In [6]:
# 5. Train and compare models

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=5,
        restore_best_weights=True
    )
]

models = {
    "GRU": build_gru_model(),
    "Stacked_GRU": build_stacked_gru_model(),
    "LSTM": build_lstm_model(),
    "Conv1D_GRU": build_conv_gru_model()
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    history = model.fit(
        train_dataset,
        epochs=10,
        validation_data=val_dataset,
        callbacks=callbacks,
        verbose=1
    )

    best_val_mae = min(history.history["val_mae"])
    test_loss, test_mae = model.evaluate(test_dataset, verbose=0)

    results[name] = {
        "model": model,
        "best_val_mae": best_val_mae,
        "test_mae": test_mae
    }

    print(f"{name} -> Best Validation MAE: {best_val_mae:.4f}, Test MAE: {test_mae:.4f}")


Training GRU...
Epoch 1/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 43s 51ms/step - loss: 21.2125 - mae: 3.3321 - val_loss: 9.7688 - val_mae: 2.4179
Epoch 2/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 50s 61ms/step - loss: 9.3800 - mae: 2.3934 - val_loss: 9.7930 - val_mae: 2.3850
Epoch 3/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 53s 65ms/step - loss: 8.8096 - mae: 2.3169 - val_loss: 9.7234 - val_mae: 2.3893
Epoch 4/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 50s 61ms/step - loss: 8.2838 - mae: 2.2458 - val_loss: 9.9522 - val_mae: 2.4137
Epoch 5/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 113s 138ms/step - loss: 7.7776 - mae: 2.1801 - val_loss: 10.1671 - val_mae: 2.4468
Epoch 6/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 42s 51ms/step - loss: 7.2887 - mae: 2.1141 - val_loss: 10.3507 - val_mae: 2.4828
Epoch 7/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 45s 55ms/step - loss: 6.8577 - mae: 2.0516 - val_loss: 10.9735 - val_mae: 2.5464
GRU -> Best Validation MAE: 2.3850, Test MAE: 2.5128

Training Stacked_GRU...
Epoch 1/10
819/819 ━━━━━━━━━━━━━━━━━━━━ 155s 187ms/step - l